# Unsupervised Text Vectorization & Feature Extraction Pipeline

This notebook encapsulates the **Feature Engineering phase** of our text clustering pipeline. Having sanitised and structured our text corpus in the previous stage, our objective now is to translate raw tokens into mathematical representations suitable for geometric clustering algorithms.

### Objectives:
1. **Sparse Representation Routing:** Implement a highly optimized Term Frequency-Inverse Document Frequency (TF-IDF) configuration to capture explicit word-level features.
2. **Dense Representation Routing:** Extract deep semantic vectors utilizing contextual embedding representations (e.g., Transformer-based models).
3. **Dimensionality Profiling:** Compare structural traits (sparsity, density, variance) across both embedding families.
4. **Serialization:** Securely persist vectorized numpy matrices and sparse configurations to disk, ensuring direct injection readiness for the clustering trials phase.

---

### Project Architecture Note:
> **Experimental Phase vs. Production Code:** > This notebook is dedicated exclusively to the **Exploration and Model Trials phase** (Vector configuration tuning, embedding dimension mapping, and validation testing). 
> 
> Once the optimal vectorization strategies are finalized here, the underlying extraction logic will be refactored into modular, production-ready pipeline modules inside `/src/features/` to fuel our backend microservices for deployment.
---

# Embeddings 

## Import needed libs

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
import numpy as np

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [3]:
base_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))

path = os.path.join(
    base_dir,
    "data/processed/train_processed.csv"
)
df_train = pd.read_csv(path)

display(df_train.head())
print(df_train.columns)
print(df_train.shape)

,text,target,target_name,word_count_raw,cleaned_text,word_count_cleaned
0,"No doubt this is an old question, but I didn't...",5,comp.windows.x,74,doubt old question find answer faq could find ...,31
1,"Try this one, a favorite around here:\n\nBurea...",16,talk.politics.guns,24,try one favorite around bureau asshole tightwa...,14
2,\nA 68070 is just a 68010 with a built in MMU....,1,comp.graphics,26,built mmu even think moto manufacture ian roma...,11
3,Last two copies of silverlining 5.42 from La C...,6,misc.forsale,58,last two copy silverlining cie sale disk manag...,42
4,\n[... other info deleted ...]\n\n\nAre you su...,4,comp.sys.mac.hardware,135,info deleted sure problem caused software seen...,67


Index(['text', 'target', 'target_name', 'word_count_raw', 'cleaned_text',
       'word_count_cleaned'],
      dtype='object')
(14518, 6)


In [4]:
path_val = os.path.join(
    base_dir,
    "data/processed/val_processed.csv"
)
df_val = pd.read_csv(path_val)


path_test = os.path.join(
    base_dir,
    "data/processed/test_processed.csv"
)
df_test = pd.read_csv(path_test)


## 1. Baseline: TF-IDF Embedding

## Sparse Representation Routing (TF-IDF Pipeline & Serialization)

In this phase, we execute our first vectorization strategy using a statistical **Term Frequency-Inverse Document Frequency (TF-IDF)** framework. This pipeline extracts explicit word-level features to serve as our baseline sparse representation.

#### Hyperparameter Configuration Rationale:
* `max_features=10000`: Caps the vocabulary matrix at the top 10,000 most informative tokens. This provides a hard boundary against the curse of dimensionality while retaining critical domain-specific terms.
* `max_df=0.95`: Automatically prunes hyper-frequent words appearing in more than 95% of documents, filtering out residual structural stopwords.
* `min_df=2`: Excludes rare words/typos that appear only once across the entire corpus, reducing noise and preventing matrix inflation.

---

#### Dimensionality Profiling & Architectural Notes:

##### 1. Preventing Data Leakage (The Split Constraint)
We explicitly restrict the vocabulary building process (`.fit_transform()`) to the **Training Set (`df_train`)** only. The Validation and Test text arrays are mapped strictly via `.transform()`. 

##### 2. Structural & Sparsity Traits (Matrix Profiling)
* **Dimensions:** The training sequence produces a data matrix of shape $(14518, 10000)$.
* **Sparsity Factor:** Although the matrix allocates 10,000 potential feature dimensions per document, any single 74-word article will only contain non-zero values for its specific keywords. Consequently, this matrix is over **$99\%$ sparse (filled with zeros)**, packed natively inside a compressed `scipy.sparse.csr_matrix` format to optimize RAM.

##### 3. Serialization (`joblib` Layer)
Because vectorization and token tracking require significant calculation overhead, we enforce **Serialization**. By writing the fitted vectorizer configuration alongside the partitioned matrices directly into binary files (`.pkl`)

In [5]:
X_train_text = df_train['cleaned_text'].fillna("").astype(str)

tfidf_vectorizer = TfidfVectorizer(max_df=0.95, min_df=2, max_features=10000)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_text)

print(f"TF-IDF Matrix Shape: {X_train_tfidf.shape}")
print(f"Vocabulary Size: {len(tfidf_vectorizer.vocabulary_)} words")

TF-IDF Matrix Shape: (14518, 10000)
Vocabulary Size: 10000 words


In [6]:
print(type(X_train_tfidf))

<class 'scipy.sparse._csr.csr_matrix'>


In [8]:
features_path = os.path.join(base_dir, "data/processed/")

joblib.dump(tfidf_vectorizer, os.path.join(features_path, "tfidf_vectorizer.pkl"))

joblib.dump(X_train_tfidf, os.path.join(features_path, "train_tfidf_matrix.pkl"))

print("Saved 'tfidf_vectorizer.pkl' and 'train_tfidf_matrix.pkl' successfully!")

Saved 'tfidf_vectorizer.pkl' and 'train_tfidf_matrix.pkl' successfully!


In [9]:
X_val_text = df_val['cleaned_text'].fillna("").astype(str)
X_test_text = df_test['cleaned_text'].fillna("").astype(str)

X_val_tfidf = tfidf_vectorizer.transform(X_val_text)
X_test_tfidf = tfidf_vectorizer.transform(X_test_text)

print(f"Validation TF-IDF Shape: {X_val_tfidf.shape}")
print(f"Test TF-IDF Shape:       {X_test_tfidf.shape}")

joblib.dump(X_val_tfidf, os.path.join(features_path, "val_tfidf_matrix.pkl"))
joblib.dump(X_test_tfidf, os.path.join(features_path, "test_tfidf_matrix.pkl"))

print("Saved 'val_tfidf_matrix.pkl' and 'test_tfidf_matrix.pkl' successfully!")

Validation TF-IDF Shape: (1819, 10000)
Test TF-IDF Shape:       (1817, 10000)
Saved 'val_tfidf_matrix.pkl' and 'test_tfidf_matrix.pkl' successfully!


## 2. Advanced: Sentence Transformers (BERT-based) Embeddings

## Dense Representation Routing (Contextual Embeddings Pipeline)

In this phase, we implement our second vectorization vector strategy using deep learning representations. Unlike the statistical word-counting nature of TF-IDF, we employ a pre-trained Transformer model—**`all-MiniLM-L6-v2`**—from the Sentence-Transformers library to map documents into a continuous, dense vector space.

#### MLOps & Production Safeguards:
1. **Model Localization (`model.save`)**: We pull the model from Hugging Face once, then immediately cache and serialize the model architecture into `../models/all-MiniLM-L6-v2`. This eliminates external API dependencies, guarantees 100% offline reproducibility, and speeds up kernel restarts.
2. **Batch Optimization**: Text sequences are converted into native Python lists and fed into the transformer using a chunked execution layout (`batch_size=32`). 
---

#### Dimensionality Profiling: Sparse vs. Dense Matrices

Now that both vectorization routes are fully executed, we contrast the structural and mathematical traits of our feature spaces before pushing them to our clustering models.

| Metric / Structural Trait | Sparse Matrix Route (TF-IDF) | Dense Matrix Route (BERT / MiniLM) |
| :--- | :--- | :--- |
| **Mathematical Structure** | `scipy.sparse.csr_matrix` (Compressed) | `numpy.ndarray` (Continuous Float Matrix) |
| **Final Shape (Train)** | $(14518, 10000)$ | $(14518, 384)$ |
| **Feature Dimensionality** | **High-Dimensional** (10,000 explicit tokens) | **Low-Dimensional** (384 latent semantic dimensions) |
| **Sparsity vs. Density** | **$>99\%$ Sparse** (Dominated by structural zeros) | **$100\%$ Dense** (Every single coordinate holds a real float weight) |
| **Semantic Capture** | Strict keyword matching. Misses synonyms and contextual relationships. | Deep contextual awareness. Captures intent, phrasing variations, and semantic proximity. |

##### Architectural Implications for Downstream Clustering:
* **The Distance Metric Impact:** Standard distance metrics like *Euclidean Distance* (used heavily by K-Means) break down in high-dimensional spaces like our TF-IDF matrix because distances between points begin to collapse and look identical (Curse of Dimensionality). TF-IDF clustering will likely require strict cosine similarity or prior dimensionality reduction (like TruncatedSVD).
* **The Dense Space Advantage:** Our BERT matrix compresses the entire document context into just **384 continuous dimensions** while maintaining a 100% dense layout. This tight, continuous boundary is highly optimal for geometric clustering algorithms and density-based estimators (like GMM or HDBSCAN) because it preserves localized variance without the noise of empty spatial dimensions.

---

#### 3. Feature Serialization (`joblib` Layer)
To preserve these expensive deep-learning matrix outputs, we serialize the encoded arrays (`X_train_bert`, `X_val_bert`, `X_test_bert`) into compressed binary storage (`.pkl`). 

In [ ]:
print("Loading Sentence Transformer model...")
model = SentenceTransformer('all-MiniLM-L6-v2')



In [ ]:
model.save('../../models/all-MiniLM-L6-v2')
print("Model saved locally!")

In [11]:
model = SentenceTransformer('../models/all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [12]:
X_train_text = df_train['cleaned_text'].fillna("").astype(str).tolist()
X_val_text = df_val['cleaned_text'].fillna("").astype(str).tolist()
X_test_text = df_test['cleaned_text'].fillna("").astype(str).tolist()

print("Encoding Train set...")
X_train_bert = model.encode(X_train_text, show_progress_bar=True, batch_size=32)

print("Encoding Validation set...")
X_val_bert = model.encode(X_val_text, show_progress_bar=True, batch_size=32)

print("Encoding Test set...")
X_test_bert = model.encode(X_test_text, show_progress_bar=True, batch_size=32)

print(f"Train BERT Matrix Shape: {X_train_bert.shape}")
print(f"Val BERT Matrix Shape:   {X_val_bert.shape}")
print(f"Test BERT Matrix Shape:  {X_test_bert.shape}")

Encoding Train set...


Batches:   0%|          | 0/454 [00:00<?, ?it/s]

Encoding Validation set...


Batches:   0%|          | 0/57 [00:00<?, ?it/s]

Encoding Test set...


Batches:   0%|          | 0/57 [00:00<?, ?it/s]

Train BERT Matrix Shape: (14518, 384)
Val BERT Matrix Shape:   (1819, 384)
Test BERT Matrix Shape:  (1817, 384)


In [13]:
joblib.dump(X_train_bert, os.path.join(features_path, "train_bert_matrix.pkl"))
joblib.dump(X_val_bert, os.path.join(features_path, "val_bert_matrix.pkl"))
joblib.dump(X_test_bert, os.path.join(features_path, "test_bert_matrix.pkl"))

print("Saved all BERT matrices ('train_bert_matrix.pkl', etc.) successfully!")

Saved all BERT matrices ('train_bert_matrix.pkl', etc.) successfully!
